# Audit the experience used to train the cost adapter

Use a **CPU runtime** and choose **Runtime → Run all**. This notebook reads the 40 saved collection episodes. It performs no simulator rollouts, model loading, or training. The original pilot output is left unchanged; derived results are written under `experience_audit/run_0`.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys

drive.mount('/content/drive')
REPO_URL = 'https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_REF = 'curriculum-cost-adapter'
REPO = Path('/content/i-jepa-wm-experience-audit')
PILOT = Path('/content/drive/MyDrive/datasets/curriculum_cost_adapter/pilot_v1')

def run(command, cwd=None):
    print('$', ' '.join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

required = [PILOT / 'manifest.json', PILOT / 'runtime.json', PILOT / 'stages/run_0/collect.json']
missing = [str(path) for path in required if not path.is_file()]
assert not missing, 'Missing pilot files: ' + ', '.join(missing)
if not REPO.exists():
    run(['git', 'clone', '--branch', SOURCE_REF, '--single-branch', REPO_URL, REPO])
else:
    assert (REPO / '.git').is_dir(), f'{REPO} exists but is not a git clone'
    assert not subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip(), 'Audit source clone has local changes'
    run(['git', 'fetch', 'origin', SOURCE_REF], cwd=REPO)
    run(['git', 'checkout', '--detach', 'origin/' + SOURCE_REF], cwd=REPO)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
print('Audit source commit:', commit)


## Run the read-only analysis

The command validates that every saved state, action and frame belongs to the recorded run. It then reproduces the pair filter used during adapter fitting. A 0.25-unit tolerance separates clear progress or regression from tiny changes.


In [ ]:
run([sys.executable, '-m', 'src.mylonas_ijepa_wm.audit_curriculum_experience', '--pilot-dir', PILOT], cwd=REPO)
REPORT_DIR = PILOT / 'experience_audit/run_0'
summary = json.loads((REPORT_DIR / 'summary.json').read_text())
print('\nHEADLINE RESULTS')
print(f"Episodes: {summary['episodes']} successful: {summary['successful_episodes']}")
print(f"Primitive actions: {summary['primitive_actions']} frames: {summary['frames']}")
print(f"Actions clearly closer: {summary['motion']['closer_fraction']:.1%}")
print(f"Actions clearly away: {summary['motion']['away_fraction']:.1%}")
print(f"Mean successful path efficiency: {summary['mean_success_path_efficiency']:.1%}")
pairs = summary['pair_quality']['after']['episode_uniform']
print(f"Training pairs useful: {pairs['progress_fraction']:.1%}")
print(f"Training pairs unclear: {pairs['ambiguous_fraction']:.1%}")
print(f"Training pairs contradictory: {pairs['contradictory_fraction']:.1%}")
print('Cases lost after adaptation:', [x['case_id'] for x in summary['evaluation']['lost_cases']])
print('Saved filter counts verified:', summary['saved_filter_verified'])
print('\nTRAINING-LOSS HISTORY (optimization behavior, not accuracy)')
print(json.dumps(summary['loss_history'], indent=2))
print('\nEXISTING CANDIDATE AUDITS')
print('These predefined diagnostic cases may differ from the dynamically identified lost cases above.')
if summary['existing_candidate_audits']:
    for audit in summary['existing_candidate_audits']:
        print(json.dumps(audit, indent=2))
else:
    print('No saved candidate audits were found.')


## Inspect the episode table and figures

A good episode usually has high path efficiency, few away steps, and a mostly decreasing distance curve. The pair chart is more direct for training: red means the later image was actually farther from the goal even though training treated it as preferable. Gray pairs changed too little to establish a confident ordering.


In [ ]:
import pandas as pd
from IPython.display import display, Image, Markdown

episodes = pd.read_csv(REPORT_DIR / 'episodes.csv')
columns = ['case_id', 'steps', 'success_path_efficiency', 'away_fraction', 'cumulative_distance_increase', 'action_reversal_fraction', 'pairs_after_contradictory_fraction']
display(episodes[columns].sort_values(['pairs_after_contradictory_fraction', 'away_fraction'], ascending=False, na_position='last'))
figures = [
    ('progress.png', 'Every distance curve; upward segments move away from the goal.'),
    ('pair_quality.png', 'The exact supervision before and after revisit filtering.'),
    ('coverage.png', 'Where the collection trajectories actually went.'),
    ('episode_examples.png', 'Geometry for selected weak and strong episodes.'),
    ('frames.png', 'Real start, middle, and end observations from those episodes.'),
    ('training_pair_examples.png', 'The strongest surviving contradictory pairs, if any.'),
    ('lost_cases.png', 'Raw versus adapted paths for evaluation cases lost after fitting.'),
]
for filename, explanation in figures:
    display(Markdown('### ' + explanation))
    display(Image(filename=str(REPORT_DIR / filename)))


## What this audit cannot answer

The original goal embedding was not saved, so this CPU audit cannot recompute the adapter's exact cost ordering. It does not use a terminal success frame as a substitute. The existing candidate-audit numbers remain in `summary.json`. After running, download this executed notebook and share it for interpretation.


In [ ]:
print((REPORT_DIR / 'README.md').read_text())
print('\nAll derived files:', REPORT_DIR)
print('In Colab, choose File → Download → Download .ipynb to share the executed audit.')
